# GO Terms parsing for the v1 of the dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:
go_terms = pd.read_csv('train_terms.tsv', sep='\t')
print(len(go_terms))

5363863


In [3]:
go_terms

,EntryID,term,aspect
0,A0A009IHW8,GO:0008152,BPO
1,A0A009IHW8,GO:0034655,BPO
2,A0A009IHW8,GO:0072523,BPO
3,A0A009IHW8,GO:0044270,BPO
4,A0A009IHW8,GO:0006753,BPO
...,...,...,...
5363858,X5L565,GO:0050649,MFO
5363859,X5L565,GO:0016491,MFO
5363860,X5M5N0,GO:0005515,MFO
5363861,X5M5N0,GO:0005488,MFO


In [4]:
go_terms = go_terms.rename(columns={'EntryID':'protein_id', 'term':'go_id', 'aspect': 'go_aspect'})

In [5]:
from tqdm import tqdm
import pandas as pd

def collapse_go_terms_per_protein(df):
    go_fields = ["go_id", "go_aspect"]

    def agg_go_group(group):
        first = group.iloc[0].to_dict()

        go_ids = group["go_id"].tolist()  # Keep original list
        go_bp = group.loc[group["go_aspect"] == "BPO", "go_id"].tolist()
        go_mf = group.loc[group["go_aspect"] == "MFO", "go_id"].tolist()
        go_cc = group.loc[group["go_aspect"] == "CCO", "go_id"].tolist()

        out = {
            "protein_id": first["protein_id"],
            "go_ids": go_ids,
            "go_bp": go_bp if go_bp else np.nan,
            "go_mf": go_mf if go_mf else np.nan,
            "go_cc": go_cc if go_cc else np.nan,
        }

        # Add other fields from first row
        for col in df.columns:
            if col not in go_fields + ["protein_id"]:
                out[col] = first[col]

        return pd.Series(out)

    tqdm.pandas(desc="Collapsing GO terms")
    collapsed_df = df.groupby("protein_id").progress_apply(agg_go_group).reset_index(drop=True)
    return collapsed_df

In [6]:
go_terms = collapse_go_terms_per_protein(go_terms)

Collapsing GO terms: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 142246/142246 [01:55<00:00, 1234.89it/s]


In [7]:
go_terms

,protein_id,go_ids,go_bp,go_mf,go_cc
0,A0A009IHW8,"[GO:0008152, GO:0034655, GO:0072523, GO:004427...","[GO:0008152, GO:0034655, GO:0072523, GO:004427...","[GO:0003674, GO:0003953, GO:0016787, GO:001679...",NaN
1,A0A021WW32,"[GO:0048869, GO:0048856, GO:0022008, GO:006500...","[GO:0048869, GO:0048856, GO:0022008, GO:006500...",NaN,"[GO:0099086, GO:0000228, GO:0005622, GO:004322..."
2,A0A021WZA4,"[GO:0071944, GO:0005575, GO:0110165, GO:001602...",NaN,NaN,"[GO:0071944, GO:0005575, GO:0110165, GO:001602..."
3,A0A023FBW4,"[GO:0019956, GO:0005488, GO:0003674, GO:001995...",NaN,"[GO:0019956, GO:0005488, GO:0003674, GO:001995...",NaN
4,A0A023FBW7,"[GO:0019957, GO:0019956, GO:0005488, GO:000367...",NaN,"[GO:0019957, GO:0019956, GO:0005488, GO:000367...",NaN
...,...,...,...,...,...
142241,X6RKS3,"[GO:0005622, GO:0031090, GO:0043226, GO:003196...",NaN,NaN,"[GO:0005622, GO:0031090, GO:0043226, GO:003196..."
142242,X6RLN4,"[GO:0005737, GO:0005829, GO:0005575, GO:000562...",NaN,NaN,"[GO:0005737, GO:0005829, GO:0005575, GO:000562..."
142243,X6RLP6,"[GO:0005622, GO:0031981, GO:0043229, GO:004322...",NaN,NaN,"[GO:0005622, GO:0031981, GO:0043229, GO:004322..."
142244,X6RLR1,"[GO:0005622, GO:0031981, GO:0043226, GO:000573...",NaN,NaN,"[GO:0005622, GO:0031981, GO:0043226, GO:000573..."


In [8]:
IA = pd.read_csv('IA.txt',sep='\t', names = ['go_id','go_weights'])
IA = IA.set_index("go_id")
IA

,go_weights
go_id,
GO:0000001,0.000000
GO:0000002,3.103836
GO:0000003,3.439404
GO:0000011,0.056584
GO:0000012,6.400377
...,...
GO:2001083,7.159871
GO:2001084,7.592457
GO:2001085,7.159871


# CAFA5 Test File

In [9]:
!seqkit fx2tab testsuperset.fasta > testsuperset.tsv
# !testsuperset.tsv

In [10]:
test_proteins = pd.read_csv('testsuperset.tsv', sep = '\t', names = ['protein_id', 'tax_id','sequence', 'temp'])
test_proteins = test_proteins.drop(columns=['temp'])
test_proteins = test_proteins.drop_duplicates()
test_proteins = test_proteins.set_index('protein_id')
test_proteins

,tax_id,sequence
protein_id,,
Q9CQV8,10090,MTMDKSELVQKAKLAEQAERYDDMAAAMKAVTEQGHELSNEERNLL...
P62259,10090,MDDREDLVYQAKLAEQAERYDEMVESMKKVAGMDVELTVEERNLLS...
P68510,10090,MGDREQLLQRARLAEQAERYDDMASAMKAVTELNEPLSNEDRNLLS...
P61982,10090,MVDREQLVQKARLAEQAERYDDMAAAMKNVTELNEPLSNEERNLLS...
O70456,10090,MERASLIQKAKLAEQAERYEDMAAFMKSAVEKGEELSCEERNLLSV...
...,...,...
P08380,38281,GNCKCDDEGPNVRTAPLTGYVDLGYCNEGWEKCASYYSPIAECCRKKK
C0HK72,38281,RGICLEPKVVGPCKARIRRFYYDSETGKCTPFIYGGCGGNGNNFET...
C0HK73,38281,GSICLEPKVVGPCKAGIRRFYFDSETGKCTLFLYGGCKGNGNNFET...


In [11]:
test_tax = pd.read_csv('testsuperset-taxon-list.tsv', sep='\t')
test_tax = test_tax.set_index('ID')
test_tax

,Species
ID,
9606,homo sapiens[All Names]
10090,mus musculus[All Names]
10116,Rattus norvegicus
3702,Arabidopsis thaliana[All Names]
83333,Escherichia coli K-12[all names]
...,...
8671,Pseudechis porphyriacus (snakes)
8673,Pseudonaja textilis (snakes)
930089,Bipolaris zeicola 26-R-13 (ascomycetes fungi)


# Protein Metadata

In [12]:
import requests
import pandas as pd
from time import sleep
from io import StringIO
from tqdm import tqdm

def fetch_uniprot_metadata_large(all_accessions, output_file="uniprot_metadata.tsv", deleted_file="deleted_accessions.txt", batch_size=500):
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    fields = [
        "accession",
        "protein_name",
        "cc_function",
        "organism_name",
        "length",
        "cc_subcellular_location",
        "sequence"
    ]

    # Prepare output
    first_batch = True
    deleted_accessions = []

    # Break into batches
    batches = [all_accessions[i:i + batch_size] for i in range(0, len(all_accessions), batch_size)]

    for batch in tqdm(batches, desc="Processing batches"):
        query = " OR ".join([f"accession:{acc}" for acc in batch])
        params = {
            "query": query,
            "format": "tsv",
            "fields": ",".join(fields),
            "size": batch_size
        }

        try:
            r = requests.get(base_url, params=params, timeout=30)
            if r.status_code != 200:
                print(f"Failed with status {r.status_code} — retrying after delay")
                sleep(5)
                continue

            df = pd.read_csv(StringIO(r.text), sep="\t")

            # Identify deleted entries
            deleted_mask = df["Protein names"].str.contains("deleted", case=False, na=False)
            deleted_accessions.extend(df.loc[deleted_mask, "Entry"].tolist())

            # Filter non-deleted
            df_clean = df[~deleted_mask]

            # Write to file incrementally
            df_clean.to_csv(output_file, sep="\t", index=False, mode='a', header=first_batch)
            first_batch = False

        except Exception as e:
            print(f"Error on batch: {e}")
            sleep(5)

        sleep(0.1)  # rate-limit buffer

    # Save deleted accessions
    with open(deleted_file, "w") as f:
        for acc in deleted_accessions:
            f.write(acc + "\n")

    print(f"✅ Done. Saved to {output_file}, deleted proteins saved to {deleted_file}")

In [9]:
accessions = go_terms['protein_id'].unique()
fetch_uniprot_metadata_large(accessions)

Processing batches: 100%|████████████████████████████████████████████████████████████████████████████████████████| 285/285 [21:59<00:00,  4.63s/it]

✅ Done. Saved to uniprot_metadata.tsv, deleted proteins saved to deleted_accessions.txt


In [15]:
fetch_uniprot_metadata_large(test_proteins['protein_id'].unique(), 
                             output_file="uniprot_metadata_test_set.tsv", deleted_file="deleted_accessions_test_set.txt")

Processing batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 284/284 [11:50<00:00,  2.50s/it]

✅ Done. Saved to uniprot_metadata_test_set.tsv, deleted proteins saved to deleted_accessions_test_set.txt


# Merging UniProt Metadata and Train File

In [12]:
metadata = pd.read_csv('uniprot_metadata.tsv',sep='\t')

In [13]:
metadata = metadata.rename(columns={'Entry':'protein_id', 'Protein names':'protein_names',
                                    'Function [CC]':'protein_function', 'Organism':'organism',
                                    'Length':'length', 'Subcellular location [CC]':'subcellular_location',
                                    'Sequence':'sequence'})

In [14]:
metadata.columns

Index(['protein_id', 'protein_names', 'protein_function', 'organism', 'length',
       'subcellular_location', 'sequence'],
      dtype='object')

In [15]:
merged = pd.merge(metadata,go_terms, on='protein_id',how='inner')

In [16]:
len(merged['protein_id'].unique())

133941

In [17]:
merged["protein_function"] = merged["protein_function"].str.replace(r"^FUNCTION:\s*", "", regex=True)
merged["subcellular_location"] = merged["subcellular_location"].str.replace(r"^SUBCELLULAR LOCATION:\s*", "", regex=True)

### Removing the protein's that have a name as 'merged'. This means that those entries were merged into other ones, so they are redundant.

In [18]:
import pandas as pd

# Identify rows to remove
merged_to_remove = merged[merged["protein_names"] == "merged"]

# Extract ProteinID values to append
to_append = merged_to_remove["protein_id"].dropna().unique().tolist()

# Remove those rows from the DataFrame
merged = merged[merged["protein_names"] != "merged"]

# Append to 'deleted_accessions.txt'
with open("deleted_accessions.txt", "a") as f:
    for acc in to_append:
        f.write(str(acc) + "\n")

print(f"✅ Removed {len(to_append)} 'merged' entries and appended to deleted_accessions.txt")

✅ Removed 445 'merged' entries and appended to deleted_accessions.txt


In [19]:
merged

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc
0,A0A087X1C5,Putative cytochrome P450 2D7 (EC 1.14.14.1),May be responsible for the metabolism of many ...,Homo sapiens (Human),515.0,Membrane {ECO:0000305}; Multi-pass membrane pr...,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,"[GO:0008152, GO:0051716, GO:0009987, GO:007146...","[GO:0008152, GO:0051716, GO:0009987, GO:007146...","[GO:0003674, GO:0070330, GO:0016705, GO:000382...","[GO:0005622, GO:0043229, GO:0043226, GO:011016..."
1,A0A0B4J2F0,Protein PIGBOS1 (PIGB opposite strand protein 1),Plays a role in regulation of the unfolded pro...,Homo sapiens (Human),54.0,Mitochondrion outer membrane {ECO:0000269|PubM...,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,"[GO:0048583, GO:0050789, GO:0010646, GO:002305...","[GO:0048583, GO:0050789, GO:0010646, GO:002305...","[GO:0005515, GO:0005488, GO:0003674]","[GO:0005622, GO:0031090, GO:0031967, GO:004322..."
2,A0A0A7EPL0,E4 SUMO-protein ligase PIAL1 (EC 2.3.2.-) (Pro...,"Together with MOM1 and PIAL2, regulates transc...",Arabidopsis thaliana (Mouse-ear cress),847.0,Nucleus {ECO:0000305}.,MVIPATSRFGFRAEFNTKEFQASCISLANEIDAAIGRNEVPGNIQE...,"[GO:0008152, GO:0036211, GO:0009628, GO:006500...","[GO:0008152, GO:0036211, GO:0009628, GO:006500...","[GO:0016740, GO:0003674, GO:0140096, GO:001978...",NaN
3,A0A0B4J1G0,Low affinity immunoglobulin gamma Fc region re...,Receptor for the invariable Fc fragment of imm...,Mus musculus (Mouse),249.0,"Cell membrane {ECO:0000269|PubMed:16039578, EC...",MWQLLLPTALVLTAFSGIQAGLQKAVVNLDPKWVRVLEEDSVTLRC...,"[GO:0006898, GO:0051234, GO:0002376, GO:004578...","[GO:0006898, GO:0051234, GO:0002376, GO:004578...","[GO:0019767, GO:0004888, GO:0038023, GO:001976...","[GO:0071944, GO:0005575, GO:0110165, GO:001602..."
4,A0A0B4J1N3,Protein GPR15LG (Protein GPR15 ligand) (Protei...,Highly cationic protein that has multiple func...,Mus musculus (Mouse),78.0,Secreted {ECO:0000250|UniProtKB:Q6UWK7}.,MRLLALSGLLCMLLLCFCIFSSEGRRHPAKSLKLRRCCHLSPRSKL...,"[GO:0009605, GO:0051716, GO:0040011, GO:000237...","[GO:0009605, GO:0051716, GO:0040011, GO:000237...","[GO:0098772, GO:0005488, GO:0140677, GO:000512...",NaN
...,...,...,...,...,...,...,...,...,...,...,...
133936,V9XTM1,Erythrocyte-binding antigen 175,NaN,Plasmodium sp. chimpanzee clade C3,631.0,NaN,FLDSRVNDRKNSSSNNGDLNNCREKRRGMKWECKKKNNTSNYVCIP...,"[GO:0005515, GO:0005488, GO:0003674]",NaN,"[GO:0005515, GO:0005488, GO:0003674]",NaN
133937,W5IDC3,Autonomous cohesin,NaN,Ruminococcus flavefaciens,201.0,NaN,MGSSSVTADLNNAVINVDEMNEAFKDVPDLEGEGAHITLSNTTAKP...,"[GO:0005515, GO:0005488, GO:0003674]",NaN,"[GO:0005515, GO:0005488, GO:0003674]",NaN
133938,Q9YWN5,VP9,NaN,Banna virus (BAV),283.0,NaN,MLSETELRALKKLSTTTSRVVGDSTLALPSNVKLSKGEVEKIAVTK...,"[GO:0042802, GO:0005488, GO:0005515, GO:0003674]",NaN,"[GO:0042802, GO:0005488, GO:0005515, GO:0003674]",NaN
133939,Q9Z7K1,Flagellar motor switch Domain/YscQ family (Typ...,NaN,Chlamydia pneumoniae (Chlamydophila pneumoniae),371.0,NaN,MAVAADSSASWLKSRNNFLSSLGKTEEQVAAPEFPKELCQHKIREK...,"[GO:0005515, GO:0005488, GO:0003674]",NaN,"[GO:0005515, GO:0005488, GO:0003674]",NaN


# Merging Metadata and Test File

In [20]:
metadata_test = pd.read_csv('uniprot_metadata_test_set.tsv',sep='\t')

In [21]:
metadata_test = metadata_test.rename(columns={'Entry':'protein_id', 'Protein names':'protein_names',
                                    'Function [CC]':'protein_function', 'Organism':'organism',
                                    'Length':'length', 'Subcellular location [CC]':'subcellular_location',
                                    'Sequence':'sequence'})

In [22]:
metadata_test.columns

Index(['protein_id', 'protein_names', 'protein_function', 'organism', 'length',
       'subcellular_location', 'sequence'],
      dtype='object')

In [23]:
metadata_test["protein_function"] = metadata_test["protein_function"].str.replace(r"^FUNCTION:\s*", "", regex=True)
metadata_test["subcellular_location"] = metadata_test["subcellular_location"].str.replace(r"^SUBCELLULAR LOCATION:\s*", "", regex=True)

### Adding back the Proteins with no metadata

In [24]:
import numpy as np

# Ensure deleted_accessions is already loaded
with open("deleted_accessions_test_set.txt", "r") as f:
    deleted_accessions = [line.strip() for line in f if line.strip()]

# Step 1: Create placeholder DataFrame first
placeholder_rows = pd.DataFrame({
    "protein_id": deleted_accessions
})

# Step 2: Map sequence from test_proteins
placeholder_rows["sequence"] = placeholder_rows["protein_id"].map(test_proteins["sequence"])

# Step 3: Map tax_id from test_proteins and then organism from test_tax
placeholder_rows["tax_id"] = placeholder_rows["protein_id"].map(test_proteins["tax_id"])
placeholder_rows["organism"] = placeholder_rows["tax_id"].map(test_tax["Species"])

# Step 4: Fill remaining fields with NA
placeholder_rows["protein_names"] = np.nan
placeholder_rows["protein_function"] = np.nan
placeholder_rows["length"] = np.nan
placeholder_rows["subcellular_location"] = np.nan

# Step 5: Drop tax_id since it's no longer needed
placeholder_rows = placeholder_rows.drop(columns=["tax_id"])

# Step 6: Add to metadata_test
metadata_test = pd.concat([metadata_test, placeholder_rows], ignore_index=True)

print(f"✅ Final metadata_test has {len(metadata_test)} rows.")

✅ Final metadata_test has 141864 rows.


In [25]:
metadata_test['go_ids'] = [['None']] * len(metadata_test)
metadata_test['go_bp'] = [['None']] * len(metadata_test)
metadata_test['go_mf'] = [['None']] * len(metadata_test)
metadata_test['go_cc'] = [['None']] * len(metadata_test)

In [26]:
metadata_test

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc
0,E9Q4Z2,Acetyl-CoA carboxylase 2 (EC 6.4.1.2) (ACC-beta),Mitochondrial enzyme that catalyzes the carbox...,Mus musculus (Mouse),2448.0,Mitochondrion {ECO:0000269|PubMed:10677481}.,MVLLLFLTCLVFSCLTFSWLKIWGKMTDSKPLTNSKVEANLLSSEE...,[None],[None],[None],[None]
1,E9Q876,Glucosylceramide transporter ABCA12 (EC 7.6.2....,Transports lipids such as glucosylceramides fr...,Mus musculus (Mouse),2595.0,"Cytoplasmic vesicle, secretory vesicle membran...",MASQFHQLRILVWKNWLGVKRQPLWTLVLILWPVIIFIILAITRTK...,[None],[None],[None],[None]
2,O35598,Disintegrin and metalloproteinase domain-conta...,Transmembrane metalloprotease which mediates t...,Mus musculus (Mouse),749.0,"Cell membrane {ECO:0000269|PubMed:23676497, EC...",MVLPTVLILLLSWAAGLGGQYGNPLNKYIRHYEGLSYNVDSLHQKH...,[None],[None],[None],[None]
3,O88990,Alpha-actinin-3 (Alpha-actinin skeletal muscle...,F-actin cross-linking protein which is thought...,Mus musculus (Mouse),900.0,NaN,MMMVMQPEGLGAGEGPFSGGGGGEYMEQEEDWDRDLLLDPAWEKQQ...,[None],[None],[None],[None]
4,P02716,Acetylcholine receptor subunit delta,"After binding acetylcholine, the AChR responds...",Mus musculus (Mouse),520.0,Postsynaptic cell membrane; Multi-pass membran...,MAGPVLTLGLLAALVVCALPGSWGLNEEQRLIQHLFNEKGYDKDLR...,[None],[None],[None],[None]
...,...,...,...,...,...,...,...,...,...,...,...
141859,Q4A3R3,NaN,NaN,Sus scrofa,NaN,NaN,MGTSAVILEICLLLSQVLTTVSSTTQTESTTEDRTQITETAFWETQ...,[None],[None],[None],[None]
141860,Q5E9L2,NaN,NaN,Bos taurus (cattle),NaN,NaN,MNWRFVELLYFLFVWGRISVQPSHQEPAATDQHVSKEFDWLISDRG...,[None],[None],[None],[None]
141861,Q3T0K9,NaN,NaN,Bos taurus (cattle),NaN,NaN,MNSKGQYPTQPTYPVQPPGNPVYPQTLHLPQAPPYTDAPPAYSELY...,[None],[None],[None],[None]
141862,Q58CX7,NaN,NaN,Bos taurus (cattle),NaN,NaN,MEPRLGPKAAALHLGWPFLLLWVSGLSYSVSSPASPSPSPVSRVRT...,[None],[None],[None],[None]


# Pulling Data from the GO terms OBO File

In [27]:
import obonet

with open('go-basic.obo', 'r') as f:
    go_basic = obonet.read_obo(f)

# Access node info
term = 'GO:0008150'
print(go_basic.nodes[term]['name'])  # biological_process

# Find parents
print(list(go_basic.predecessors(term)))

# Find children
print(list(go_basic.successors(term)))

biological_process
['GO:0000003', 'GO:0002376', 'GO:0008152', 'GO:0009987', 'GO:0016032', 'GO:0022414', 'GO:0023052', 'GO:0032501', 'GO:0032502', 'GO:0040007', 'GO:0040011', 'GO:0042592', 'GO:0043473', 'GO:0044419', 'GO:0044848', 'GO:0048511', 'GO:0048518', 'GO:0048519', 'GO:0050789', 'GO:0050896', 'GO:0051179', 'GO:0051703', 'GO:0065007', 'GO:0098754']
[]


In [28]:
go_id = 'GO:0034655'
name = go_basic.nodes[go_id]['name']
aspect = go_basic.nodes[go_id]['namespace']
definition = go_basic.nodes[go_id]['def']

In [29]:
print(f'The go_id is {go_id}, and its name is {name}, aspect is {aspect}, and definition is {definition}')

The go_id is GO:0034655, and its name is nucleobase-containing compound catabolic process, aspect is biological_process, and definition is "The chemical reactions and pathways resulting in the breakdown of nucleobases, nucleosides, nucleotides and nucleic acids." [GOC:mah]


In [30]:
import pandas as pd
import networkx as nx
import obonet
from tqdm import tqdm

# Load GO ontology
with open('go-basic.obo', 'r') as f:
    go_basic = obonet.read_obo(f)

# Define function to compute depth
def compute_go_shortest_len(go_graph, go_id):
    roots = {
        "biological_process": "GO:0008150",
        "molecular_function": "GO:0003674",
        "cellular_component": "GO:0005575"
    }

    lengths = []
    for root in roots.values():
        try:
            path_len = nx.shortest_path_length(go_graph, source=go_id, target=root)
            lengths.append(path_len)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue

    return min(lengths) if lengths else 0

# Build DataFrame of all GO terms
records = []
all_go_ids = list(go_basic.nodes)

for go_id in tqdm(all_go_ids, desc="Building GO metadata table"):
    node = go_basic.nodes[go_id]
    name = node.get("name")
    definition = node.get("def", "").replace('"', '')
    aspect = node.get("namespace")
    depth = compute_go_shortest_len(go_basic, go_id)
    weight = IA.loc[go_id]['go_weights']

    records.append({
        "go_id": go_id,
        "go_name": name,
        "go_def": definition,
        "go_aspect": aspect,
        "go_depth": depth, 
        "go_weight" : weight
    })

go_df = pd.DataFrame(records)

go_df_index = go_df.set_index('go_id')

Building GO metadata table: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 43248/43248 [00:05<00:00, 7624.03it/s]


In [31]:
go_df

,go_id,go_name,go_def,go_aspect,go_depth,go_weight
0,GO:0000001,mitochondrion inheritance,"The distribution of mitochondria, including th...",biological_process,5,0.000000
1,GO:0000002,mitochondrial genome maintenance,The maintenance of the structure and integrity...,biological_process,6,3.103836
2,GO:0000003,reproduction,The production of new individuals that contain...,biological_process,1,3.439404
3,GO:0000006,high-affinity zinc transmembrane transporter a...,Enables the transfer of zinc ions (Zn2+) from ...,molecular_function,5,4.409391
4,GO:0000007,low-affinity zinc ion transmembrane transporte...,Enables the transfer of a solute or solutes fr...,molecular_function,5,5.409391
...,...,...,...,...,...,...
43243,GO:2001313,UDP-4-deoxy-4-formamido-beta-L-arabinopyranose...,The chemical reactions and pathways involving ...,biological_process,4,4.129283
43244,GO:2001314,UDP-4-deoxy-4-formamido-beta-L-arabinopyranose...,The chemical reactions and pathways resulting ...,biological_process,5,0.000000
43245,GO:2001315,UDP-4-deoxy-4-formamido-beta-L-arabinopyranose...,The chemical reactions and pathways resulting ...,biological_process,5,0.000000
43246,GO:2001316,kojic acid metabolic process,The chemical reactions and pathways involving ...,biological_process,4,0.321928


In [32]:
def sort_go_terms_by_depth(df, go_df_index):
    # Convert the go_depth column into a dict for fast lookup
    go_depth_dict = go_df_index["go_depth"].to_dict()

    def sort_terms(term_list):
        if isinstance(term_list, list) and term_list:
            return sorted(term_list, key=lambda go: go_depth_dict.get(go, float('inf')))
        return np.nan

    # Sort each GO term list column by depth
    for col in ['go_bp', 'go_mf', 'go_cc']:
        df[col] = df[col].apply(sort_terms)

    return df.reset_index(drop=True)

merged = sort_go_terms_by_depth(merged, go_df_index)
merged = merged.reset_index(drop=True)

## All DFs

In [33]:
merged

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc
0,A0A087X1C5,Putative cytochrome P450 2D7 (EC 1.14.14.1),May be responsible for the metabolism of many ...,Homo sapiens (Human),515.0,Membrane {ECO:0000305}; Multi-pass membrane pr...,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,"[GO:0008152, GO:0051716, GO:0009987, GO:007146...","[GO:0008150, GO:0008152, GO:0009987, GO:005089...","[GO:0003674, GO:0003824, GO:0016491, GO:001670...","[GO:0005575, GO:0110165, GO:0005622, GO:004322..."
1,A0A0B4J2F0,Protein PIGBOS1 (PIGB opposite strand protein 1),Plays a role in regulation of the unfolded pro...,Homo sapiens (Human),54.0,Mitochondrion outer membrane {ECO:0000269|PubM...,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,"[GO:0048583, GO:0050789, GO:0010646, GO:002305...","[GO:0008150, GO:0050789, GO:0065007, GO:004858...","[GO:0003674, GO:0005488, GO:0005515]","[GO:0005575, GO:0110165, GO:0005622, GO:004322..."
2,A0A0A7EPL0,E4 SUMO-protein ligase PIAL1 (EC 2.3.2.-) (Pro...,"Together with MOM1 and PIAL2, regulates transc...",Arabidopsis thaliana (Mouse-ear cress),847.0,Nucleus {ECO:0000305}.,MVIPATSRFGFRAEFNTKEFQASCISLANEIDAAIGRNEVPGNIQE...,"[GO:0008152, GO:0036211, GO:0009628, GO:006500...","[GO:0008150, GO:0008152, GO:0065007, GO:004851...","[GO:0003674, GO:0003824, GO:0016740, GO:014009...",NaN
3,A0A0B4J1G0,Low affinity immunoglobulin gamma Fc region re...,Receptor for the invariable Fc fragment of imm...,Mus musculus (Mouse),249.0,"Cell membrane {ECO:0000269|PubMed:16039578, EC...",MWQLLLPTALVLTAFSGIQAGLQKAVVNLDPKWVRVLEEDSVTLRC...,"[GO:0006898, GO:0051234, GO:0002376, GO:004578...","[GO:0008150, GO:0002376, GO:0009987, GO:006500...","[GO:0003674, GO:0060089, GO:0038023, GO:000488...","[GO:0005575, GO:0110165, GO:0071944, GO:001602..."
4,A0A0B4J1N3,Protein GPR15LG (Protein GPR15 ligand) (Protei...,Highly cationic protein that has multiple func...,Mus musculus (Mouse),78.0,Secreted {ECO:0000250|UniProtKB:Q6UWK7}.,MRLLALSGLLCMLLLCFCIFSSEGRRHPAKSLKLRRCCHLSPRSKL...,"[GO:0009605, GO:0051716, GO:0040011, GO:000237...","[GO:0008150, GO:0040011, GO:0002376, GO:000998...","[GO:0003674, GO:0098772, GO:0005488, GO:014067...",NaN
...,...,...,...,...,...,...,...,...,...,...,...
133491,V9XTM1,Erythrocyte-binding antigen 175,NaN,Plasmodium sp. chimpanzee clade C3,631.0,NaN,FLDSRVNDRKNSSSNNGDLNNCREKRRGMKWECKKKNNTSNYVCIP...,"[GO:0005515, GO:0005488, GO:0003674]",NaN,"[GO:0003674, GO:0005488, GO:0005515]",NaN
133492,W5IDC3,Autonomous cohesin,NaN,Ruminococcus flavefaciens,201.0,NaN,MGSSSVTADLNNAVINVDEMNEAFKDVPDLEGEGAHITLSNTTAKP...,"[GO:0005515, GO:0005488, GO:0003674]",NaN,"[GO:0003674, GO:0005488, GO:0005515]",NaN
133493,Q9YWN5,VP9,NaN,Banna virus (BAV),283.0,NaN,MLSETELRALKKLSTTTSRVVGDSTLALPSNVKLSKGEVEKIAVTK...,"[GO:0042802, GO:0005488, GO:0005515, GO:0003674]",NaN,"[GO:0003674, GO:0005488, GO:0005515, GO:0042802]",NaN
133494,Q9Z7K1,Flagellar motor switch Domain/YscQ family (Typ...,NaN,Chlamydia pneumoniae (Chlamydophila pneumoniae),371.0,NaN,MAVAADSSASWLKSRNNFLSSLGKTEEQVAAPEFPKELCQHKIREK...,"[GO:0005515, GO:0005488, GO:0003674]",NaN,"[GO:0003674, GO:0005488, GO:0005515]",NaN


In [34]:
metadata_test

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc
0,E9Q4Z2,Acetyl-CoA carboxylase 2 (EC 6.4.1.2) (ACC-beta),Mitochondrial enzyme that catalyzes the carbox...,Mus musculus (Mouse),2448.0,Mitochondrion {ECO:0000269|PubMed:10677481}.,MVLLLFLTCLVFSCLTFSWLKIWGKMTDSKPLTNSKVEANLLSSEE...,[None],[None],[None],[None]
1,E9Q876,Glucosylceramide transporter ABCA12 (EC 7.6.2....,Transports lipids such as glucosylceramides fr...,Mus musculus (Mouse),2595.0,"Cytoplasmic vesicle, secretory vesicle membran...",MASQFHQLRILVWKNWLGVKRQPLWTLVLILWPVIIFIILAITRTK...,[None],[None],[None],[None]
2,O35598,Disintegrin and metalloproteinase domain-conta...,Transmembrane metalloprotease which mediates t...,Mus musculus (Mouse),749.0,"Cell membrane {ECO:0000269|PubMed:23676497, EC...",MVLPTVLILLLSWAAGLGGQYGNPLNKYIRHYEGLSYNVDSLHQKH...,[None],[None],[None],[None]
3,O88990,Alpha-actinin-3 (Alpha-actinin skeletal muscle...,F-actin cross-linking protein which is thought...,Mus musculus (Mouse),900.0,NaN,MMMVMQPEGLGAGEGPFSGGGGGEYMEQEEDWDRDLLLDPAWEKQQ...,[None],[None],[None],[None]
4,P02716,Acetylcholine receptor subunit delta,"After binding acetylcholine, the AChR responds...",Mus musculus (Mouse),520.0,Postsynaptic cell membrane; Multi-pass membran...,MAGPVLTLGLLAALVVCALPGSWGLNEEQRLIQHLFNEKGYDKDLR...,[None],[None],[None],[None]
...,...,...,...,...,...,...,...,...,...,...,...
141859,Q4A3R3,NaN,NaN,Sus scrofa,NaN,NaN,MGTSAVILEICLLLSQVLTTVSSTTQTESTTEDRTQITETAFWETQ...,[None],[None],[None],[None]
141860,Q5E9L2,NaN,NaN,Bos taurus (cattle),NaN,NaN,MNWRFVELLYFLFVWGRISVQPSHQEPAATDQHVSKEFDWLISDRG...,[None],[None],[None],[None]
141861,Q3T0K9,NaN,NaN,Bos taurus (cattle),NaN,NaN,MNSKGQYPTQPTYPVQPPGNPVYPQTLHLPQAPPYTDAPPAYSELY...,[None],[None],[None],[None]
141862,Q58CX7,NaN,NaN,Bos taurus (cattle),NaN,NaN,MEPRLGPKAAALHLGWPFLLLWVSGLSYSVSSPASPSPSPVSRVRT...,[None],[None],[None],[None]


In [36]:
from datasets import Dataset

# Assuming you already split your DataFrame earlier into train_df and test_df:
train_hf = Dataset.from_pandas(merged, preserve_index=False)
test_hf = Dataset.from_pandas(metadata_test, preserve_index=False)

In [52]:
from datasets import DatasetDict

cafa_dataset = DatasetDict({
    "train": train_hf,
    "test": test_hf
})

In [53]:
cafa_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc'],
        num_rows: 133496
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc'],
        num_rows: 141864
    })
})

In [54]:
cafa_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="cafa5_reasoning",
    commit_message="Upload CAFA5 with metadata v1.2. Removed all GO Metadata and put that into a seperate config."
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/134 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/128M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/142 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/104M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/541 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/de71c3a31a46adef0203f0f6f15d9f28d9a690fc', commit_message='Upload CAFA5 with metadata v1.2. Removed all GO Metadata and put that into a seperate config.', commit_description='', oid='de71c3a31a46adef0203f0f6f15d9f28d9a690fc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

In [55]:
from datasets import Dataset

# Ensure the columns that contain dictionaries or lists are JSON-serializable
go_hf = Dataset.from_pandas(go_df, preserve_index=False)

In [56]:
from datasets import DatasetDict

cafa_dataset = DatasetDict({
    "metadata": go_hf,
})

In [57]:
cafa_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="go_metadata",
    commit_message="Upload the GO Metadata, curated from the 2023 go-basic.obo file"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/44 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/4.07M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/867 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/5d8b08a8e4378bdcc0c29f73bfff7fb28745f45e', commit_message='Upload the GO Metadata, curated from the 2023 go-basic.obo file', commit_description='', oid='5d8b08a8e4378bdcc0c29f73bfff7fb28745f45e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

## Loading the DF back to see if it works

In [58]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")

README.md:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/133496 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/141864 [00:00<?, ? examples/s]

In [59]:
#testing if it works
from datasets import load_dataset

go_ds = load_dataset("wanglab/cafa5", "go_metadata")

Generating metadata split:   0%|          | 0/43248 [00:00<?, ? examples/s]

In [60]:
ds['train'][0]

{'protein_id': 'A0A087X1C5',
 'protein_names': 'Putative cytochrome P450 2D7 (EC 1.14.14.1)',
 'protein_function': 'May be responsible for the metabolism of many drugs and environmental chemicals that it oxidizes. It may be involved in the metabolism of codeine to morphine (PubMed:15051713). However, another study could not confirm it (PubMed:18838503). {ECO:0000269|PubMed:15051713, ECO:0000269|PubMed:18838503}.',
 'organism': 'Homo sapiens (Human)',
 'length': 515.0,
 'subcellular_location': 'Membrane {ECO:0000305}; Multi-pass membrane protein {ECO:0000255}. Cytoplasm {ECO:0000305|PubMed:15051713}. Mitochondrion {ECO:0000269|PubMed:18838503}.',
 'sequence': 'MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNLLHVDFQNTPYCFDQLRRRFGDVFSLQLAWTPVVVLNGLAAVREAMVTRGEDTADRPPAPIYQVLGFGPRSQGVILSRYGPAWREQRRFSVSTLRNLGLGKKSLEQWVTEEAACLCAAFADQAGRPFRPNGLLDKAVSNVIASLTCGRRFEYDDPRFLRLLDLAQEGLKEESGFLREVLNAVPVLPHIPALAGKVLRFQKAFLTQLDELLTEHRMTWDPAQPPRDLTEAFLAKKEKAKGSPESSFNDENLRIVVGNLFLAGMVTTSTTLAWGLLLMILHLDVQRGRR

In [61]:
ds['train'][0].keys()

dict_keys(['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc'])

In [62]:
ds['test'][0]

{'protein_id': 'E9Q4Z2',
 'protein_names': 'Acetyl-CoA carboxylase 2 (EC 6.4.1.2) (ACC-beta)',
 'protein_function': 'Mitochondrial enzyme that catalyzes the carboxylation of acetyl-CoA to malonyl-CoA and plays a central role in fatty acid metabolism (By similarity). Catalyzes a 2 steps reaction starting with the ATP-dependent carboxylation of the biotin carried by the biotin carboxyl carrier (BCC) domain followed by the transfer of the carboxyl group from carboxylated biotin to acetyl-CoA (By similarity). Through the production of malonyl-CoA that allosterically inhibits carnitine palmitoyltransferase 1 at the mitochondria, negatively regulates fatty acid oxidation (PubMed:24913514). Together with its cytosolic isozyme ACACA, which is involved in de novo fatty acid biosynthesis, promotes lipid storage (PubMed:24913514). {ECO:0000250|UniProtKB:O00763, ECO:0000269|PubMed:24913514}.',
 'organism': 'Mus musculus (Mouse)',
 'length': 2448.0,
 'subcellular_location': 'Mitochondrion {ECO:0000

In [63]:
ds['test'][0].keys()

dict_keys(['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc'])

In [64]:
go_ds['metadata'][0]

{'go_id': 'GO:0000001',
 'go_name': 'mitochondrion inheritance',
 'go_def': 'The distribution of mitochondria, including the mitochondrial genome, into daughter cells after mitosis or meiosis, mediated by interactions between mitochondria and the cytoskeleton. [GOC:mcc, PMID:10873824, PMID:11389764]',
 'go_aspect': 'biological_process',
 'go_depth': 5,
 'go_weight': 0.0}

In [65]:
go_ds['metadata'][0].keys()

dict_keys(['go_id', 'go_name', 'go_def', 'go_aspect', 'go_depth', 'go_weight'])

# Adding in GO metadata summary

In [ ]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "go_metadata")

In [ ]:
ds

DatasetDict({
    metadata: Dataset({
        features: ['go_id', 'go_name', 'go_def', 'go_aspect', 'go_depth', 'go_weight'],
        num_rows: 43248
    })
})

In [ ]:
import pandas as pd
go_df = pd.read_parquet('metadata-00000-of-00001.parquet', engine='pyarrow')

In [ ]:
from datasets import Dataset

# Ensure the columns that contain dictionaries or lists are JSON-serializable
go_hf = Dataset.from_pandas(go_df, preserve_index=False)

In [ ]:
from datasets import DatasetDict

cafa_dataset = DatasetDict({
    "metadata": go_hf,
})

In [ ]:
cafa_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="go_metadata",
    commit_message="Added in Claude summarized GO Defintions"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/44 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/9.16M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/884ffc52f4adb78b2c86f843aeabc5ea889b9bab', commit_message='Added in Claude summarized GO Defintions', commit_description='', oid='884ffc52f4adb78b2c86f843aeabc5ea889b9bab', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Changing go_def_final to go_def

In [ ]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "go_metadata")

In [ ]:
ds

DatasetDict({
    metadata: Dataset({
        features: ['go_id', 'go_name', 'go_def', 'go_aspect', 'go_depth', 'go_weight', 'go_def_clean', 'go_def_clean_length', 'claude_summary', 'claude_summary_length', 'claude_reduction_pct', 'go_def_final', 'go_def_final_length', 'has_claude_summary', 'processing_method', 'processing_date'],
        num_rows: 43248
    })
})

In [ ]:
ds = ds.rename_column('go_def', 'go_def_original')
ds = ds.rename_column('go_def_final', 'go_def')
ds = ds.rename_column('go_def_final_length', 'go_def_length')

In [ ]:
ds

DatasetDict({
    metadata: Dataset({
        features: ['go_id', 'go_name', 'go_def_original', 'go_aspect', 'go_depth', 'go_weight', 'go_def_clean', 'go_def_clean_length', 'claude_summary', 'claude_summary_length', 'claude_reduction_pct', 'go_def', 'go_def_length', 'has_claude_summary', 'processing_method', 'processing_date'],
        num_rows: 43248
    })
})

In [ ]:
ds.push_to_hub(
    "wanglab/cafa5",
    config_name="go_metadata",
    commit_message="Fixed up Column names"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/44 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/9.16M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/d1828f3e95933e7e4da206e4cf479439960555b3', commit_message='Fixed up Column names', commit_description='', oid='d1828f3e95933e7e4da206e4cf479439960555b3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Fixing Protein Function Definitions to 2023/01 version

Download full Uniprot Data from https://ftp.uniprot.org/pub/databases/uniprot/previous_releases/release-2023_01/knowledgebase/knowledgebase2023_01.tar.gz

wget https://ftp.uniprot.org/pub/databases/uniprot/previous_releases/release-2023_01/knowledgebase/uniprot_sprot-only2023_01.tar.gz
tar -xvzf uniprot_sprot-only2023_01.tar.gz
pigz -d uniprot_sprot.xml.gz

Using .dat file parser https://github.com/PNNL-Comp-Mass-Spec/Uniprot-DAT-File-Parser

In [ ]:
from lxml import etree as ET
import pandas as pd
from tqdm import tqdm

ns = {'ns': 'http://uniprot.org/uniprot'}
records = []

context = ET.iterparse("/uniprot/uniprot_sprot.xml", events=("end",), tag="{http://uniprot.org/uniprot}entry")

for event, elem in tqdm(context, desc="Parsing UniProt entries", unit=" entry"):
    accession = elem.find("ns:accession", namespaces=ns)
    comment_function = elem.find("ns:comment[@type='function']/ns:text", namespaces=ns)

    records.append({
        "accession": accession.text if accession is not None else None,
        "function_comment": comment_function.text if comment_function is not None else None
    })

    elem.clear()

df = pd.DataFrame(records)

Parsing UniProt entries: 569213entry [02:29, 3803.20entry/s]


In [ ]:
df.head()

,accession,function_comment
0,P0C9F0,"Plays a role in virus cell tropism, and may be..."
1,P0C9F1,"Plays a role in virus cell tropism, and may be..."
2,P0C9F2,"Plays a role in virus cell tropism, and may be..."
3,P0C9E9,"Plays a role in virus cell tropism, and may be..."
4,Q65209,"Plays a role in virus cell tropism, and may be..."


In [ ]:
accession_to_function = dict(zip(df['accession'], df['function_comment']))

In [1]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")

In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'full_interaction_info', 'interpro_ids', 'interpro_location'],
        num_rows: 133538
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'full_interaction_info', 'interpro_ids', 'interpro_location'],
        num_rows: 264
    })
    test_superset: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'full_interaction_info', 'interpro_ids', 'interpro_location'],
      

In [ ]:
cafa5_train = ds['train'].to_pandas()
cafa5_test = ds['test'].to_pandas()

In [ ]:
cafa5_train['protein_function_2023_01'] = cafa5_train['protein_id'].apply(lambda x: accession_to_function.get(x, None))
cafa5_test['protein_function_2023_01'] = cafa5_test['protein_id'].apply(lambda x: accession_to_function.get(x, None))

In [ ]:
diff_func = cafa5_train[cafa5_train['protein_function_2023_01'] != cafa5_train['protein_function']]

In [ ]:
diff_func

,protein_id,protein_names,protein_function,organism,length,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,interpro_ids,structure_path,protein_function_2023_01
0,A0A087X1C5,Putative cytochrome P450 2D7 (EC 1.14.14.1),May be responsible for the metabolism of many ...,Homo sapiens (Human),515.0,Membrane {ECO:0000305}; Multi-pass membrane pr...,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,"[GO:0008152, GO:0051716, GO:0009987, GO:007146...","[GO:0008150, GO:0008152, GO:0009987, GO:005089...","[GO:0003674, GO:0003824, GO:0016491, GO:001670...","[GO:0005575, GO:0110165, GO:0005622, GO:004322...","[IPR001128, IPR017972, IPR002401, IPR008069, I...",AF-A0A087X1C5-F1-model_v4.cif,May be responsible for the metabolism of many ...
1,A0A0B4J2F0,Protein PIGBOS1 (PIGB opposite strand protein 1),Plays a role in regulation of the unfolded pro...,Homo sapiens (Human),54.0,Mitochondrion outer membrane {ECO:0000269|PubM...,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,"[GO:0048583, GO:0050789, GO:0010646, GO:002305...","[GO:0008150, GO:0050789, GO:0065007, GO:004858...","[GO:0003674, GO:0005488, GO:0005515]","[GO:0005575, GO:0110165, GO:0005622, GO:004322...",None,AF-A0A0B4J2F0-F1-model_v4.cif,Plays a role in regulation of the unfolded pro...
2,A0A0A7EPL0,E4 SUMO-protein ligase PIAL1 (EC 2.3.2.-) (Pro...,"Together with MOM1 and PIAL2, regulates transc...",Arabidopsis thaliana (Mouse-ear cress),847.0,Nucleus {ECO:0000305}.,MVIPATSRFGFRAEFNTKEFQASCISLANEIDAAIGRNEVPGNIQE...,"[GO:0008152, GO:0036211, GO:0009628, GO:006500...","[GO:0008150, GO:0008152, GO:0065007, GO:004851...","[GO:0003674, GO:0003824, GO:0016740, GO:014009...",None,"[IPR004181, IPR013083]",AF-A0A0A7EPL0-F1-model_v4.cif,E4-type SUMO ligase that promotes SUMO chain f...
3,A0A0B4J1G0,Low affinity immunoglobulin gamma Fc region re...,Receptor for the invariable Fc fragment of imm...,Mus musculus (Mouse),249.0,"Cell membrane {ECO:0000269|PubMed:16039578, EC...",MWQLLLPTALVLTAFSGIQAGLQKAVVNLDPKWVRVLEEDSVTLRC...,"[GO:0006898, GO:0051234, GO:0002376, GO:004578...","[GO:0008150, GO:0002376, GO:0009987, GO:006500...","[GO:0003674, GO:0060089, GO:0038023, GO:000488...","[GO:0005575, GO:0110165, GO:0071944, GO:001602...","[IPR007110, IPR036179, IPR013783, IPR050488, I...",AF-A0A0B4J1G0-F1-model_v4.cif,Receptor for the invariable Fc fragment of imm...
4,A0A0B4J1N3,Protein GPR15LG (Protein GPR15 ligand) (Protei...,Highly cationic protein that has multiple func...,Mus musculus (Mouse),78.0,Secreted {ECO:0000250|UniProtKB:Q6UWK7}.,MRLLALSGLLCMLLLCFCIFSSEGRRHPAKSLKLRRCCHLSPRSKL...,"[GO:0009605, GO:0051716, GO:0040011, GO:000237...","[GO:0008150, GO:0040011, GO:0002376, GO:000998...","[GO:0003674, GO:0098772, GO:0005488, GO:014067...",None,[IPR031713],AF-A0A0B4J1N3-F1-model_v4.cif,Highly cationic protein that has multiple func...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133491,V9XTM1,Erythrocyte-binding antigen 175,None,Plasmodium sp. chimpanzee clade C3,631.0,None,FLDSRVNDRKNSSSNNGDLNNCREKRRGMKWECKKKNNTSNYVCIP...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None,"[IPR042202, IPR008602]",AF-V9XTM1-F1-model_v4.cif,None
133492,W5IDC3,Autonomous cohesin,None,Ruminococcus flavefaciens,201.0,None,MGSSSVTADLNNAVINVDEMNEAFKDVPDLEGEGAHITLSNTTAKP...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None,None,4WKZ.pdb,None
133493,Q9YWN5,VP9,None,Banna virus (BAV),283.0,None,MLSETELRALKKLSTTTSRVVGDSTLALPSNVKLSKGEVEKIAVTK...,"[GO:0042802, GO:0005488, GO:0005515, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515, GO:0042802]",None,"[IPR015072, IPR043116]",1W9Z.pdb,None
133494,Q9Z7K1,Flagellar motor switch Domain/YscQ family (Typ...,None,Chlamydia pneumoniae (Chlamydophila pneumoniae),371.0,None,MAVAADSSASWLKSRNNFLSSLGKTEEQVAAPEFPKELCQHKIREK...,"[GO:0005515, GO:0005488, GO:0003674]",None,"[GO:0003674, GO:0005488, GO:0005515]",None,"[IPR001543, IPR036429, IPR013385]",AF-Q9Z7K1-F1-model_v4.cif,None


In [ ]:
diff_func.iloc[0]['protein_function_2023_01']

'May be responsible for the metabolism of many drugs and environmental chemicals that it oxidizes. It may be involved in the metabolism of codeine to morphine (PubMed:15051713). However, another study could not confirm it (PubMed:18838503).'

In [ ]:
cafa5_train = ds['train'].to_pandas()
cafa5_test = ds['test'].to_pandas()

In [ ]:
cafa5_train['protein_function'] = cafa5_train['protein_id'].apply(lambda x: accession_to_function.get(x, None))
cafa5_test['protein_function'] = cafa5_test['protein_id'].apply(lambda x: accession_to_function.get(x, None))

In [ ]:
from datasets import Dataset

# Assuming you already split your DataFrame earlier into train_df and test_df:
train_hf = Dataset.from_pandas(cafa5_train, preserve_index=False)
test_hf = Dataset.from_pandas(cafa5_test, preserve_index=False)

In [ ]:
from datasets import DatasetDict

cafa_dataset = DatasetDict({
    "train": train_hf,
    "test": test_hf
})

In [ ]:
cafa_dataset

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 133496
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'interpro_ids', 'structure_path'],
        num_rows: 141864
    })
})

In [ ]:
cafa_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="cafa5_reasoning",
    commit_message="Protein Functions are from 2023-01 now"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/134 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/127M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/142 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/102M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/784011d61b517c5ed6fc8b393fe2801239bec9ca', commit_message='Protein Functions are from 2023-01 now', commit_description='', oid='784011d61b517c5ed6fc8b393fe2801239bec9ca', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

## Removing PubMed ID and other annotations from function summaries

In [1]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "cafa5_reasoning")

In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'full_interaction_info', 'interpro_ids', 'interpro_location'],
        num_rows: 133538
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'full_interaction_info', 'interpro_ids', 'interpro_location'],
        num_rows: 264
    })
    test_superset: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'full_interaction_info', 'interpro_ids', 'interpro_location'],
      

In [28]:
pattern = r' \(([^)]*PubMed[^)]*)\)'

First remove with space

In [31]:
import re

# Define a safe clean-up function
def clean_fn(f):
    return re.sub(pattern, '', f).strip() if isinstance(f, str) else f

# Apply it safely to all splits
ds = ds.map(lambda x: {"protein_function": clean_fn(x["protein_function"])})

Map:   0%|          | 0/133538 [00:00<?, ? examples/s]

Map:   0%|          | 0/264 [00:00<?, ? examples/s]

Map:   0%|          | 0/141864 [00:00<?, ? examples/s]

In [36]:
pattern = r'\(([^)]*PubMed[^)]*)\)'

Then without space

In [37]:
import re

# Define a safe clean-up function
def clean_fn(f):
    return re.sub(pattern, '', f).strip() if isinstance(f, str) else f

# Apply it safely to all splits
ds = ds.map(lambda x: {"protein_function": clean_fn(x["protein_function"])})

Map:   0%|          | 0/133538 [00:00<?, ? examples/s]

Map:   0%|          | 0/264 [00:00<?, ? examples/s]

Map:   0%|          | 0/141864 [00:00<?, ? examples/s]

In [39]:
count = 0
for i in ds['train']['protein_function']:
    if isinstance(i, str) and 'PubMed' in i:
        count+=1
print(count)

33


Removed all the citations, now these 33 that are left only have direct references in the summary. For example, in the sentence the summary directly uses the citation as a noun. For example "According to PubMed:15775980, it is required for ..."

In [41]:
ds

DatasetDict({
    train: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'full_interaction_info', 'interpro_ids', 'interpro_location'],
        num_rows: 133538
    })
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'full_interaction_info', 'interpro_ids', 'interpro_location'],
        num_rows: 264
    })
    test_superset: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'full_interaction_info', 'interpro_ids', 'interpro_location'],
      

In [43]:
ds.push_to_hub(
    "wanglab/cafa5",
    config_name="cafa5_reasoning",
    commit_message="Indirect Pubmed references have been removed from Protein Functions"
)

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/67 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/104M [00:00<?, ?B/s]

Creating parquet from Arrow format:   0%|          | 0/67 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/78.0M [00:00<?, ?B/s]

...

Uploading...:   0%|          | 0.00/395k [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/142 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/156M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/d9a22e823d52d32e335ddd8b025cf1b8ab3fe434', commit_message='Indirect Pubmed references have been removed from Protein Functions', commit_description='', oid='d9a22e823d52d32e335ddd8b025cf1b8ab3fe434', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)